### Structured Output

Models can be requested to provide their response in a format matching agreement schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing . Langchain supports multiple schema types and methods for enforcing structured output .

### Pydantic

Pydantic models provide the richest feature set with field validation, description, and nested structures

In [1]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from dotenv import load_dotenv
import os

load_dotenv()
# os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")

model = ChatNVIDIA(
    model = "nvidia/nemotron-3-ultra-550b-a55b",
    api_key = os.getenv("NVIDIA_API_KEY")
)
model

ChatNVIDIA(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15', 'langchain-nvidia-ai-endpoints': '1.4.3'}}, profile={}, base_url='https://integrate.api.nvidia.com/v1', model='nvidia/nemotron-3-ultra-550b-a55b', default_headers={}, model_kwargs={})

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie rating out of 10")

In [7]:
model_structured_output = model.with_structured_output(Movie)
model_structured_output

<langchain_nvidia_ai_endpoints.chat_models.ChatNVIDIA.with_structured_output.<locals>._FallbackRunnable at 0x275bcfc4050>

In [9]:
model.invoke("Please provide the details of the movie 'Inception'")

AIMessage(content='Here is a comprehensive breakdown of the movie **Inception (2010)**.\n\n---\n\n### **Quick Facts**\n*   **Title:** Inception\n*   **Release Date:** July 16, 2010 (Premiere in London); July 16, 2010 (US)\n*   **Director:** Christopher Nolan\n*   **Writer:** Christopher Nolan\n*   **Producer:** Christopher Nolan, Emma Thomas\n*   **Cinematography:** Wally Pfister\n*   **Music:** Hans Zimmer\n*   **Editor:** Lee Smith\n*   **Production Companies:** Syncopy Inc., Legendary Pictures\n*   **Distributor:** Warner Bros. Pictures\n*   **Budget:** $160 million\n*   **Box Office:** $839 million (worldwide)\n*   **Runtime:** 148 minutes\n*   **Rating:** PG-13 (for sequences of violence and action throughout)\n\n---\n\n### **Principal Cast**\n| Actor | Character | Role Description |\n| :--- | :--- | :--- |\n| **Leonardo DiCaprio** | **Dom Cobb** | A skilled "Extractor" who steals secrets from the subconscious. Haunted by the death of his wife. |\n| **Marion Cotillard** | **Mal Co

In [10]:
model_structured_output.invoke("Please provide the details of the movie 'Inception'")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")


In [ ]:
'''ChatNVIDIA has not implemented include_raw option'''
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="This year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie rating out of 10")

model_structured_output = model.with_structured_output(Movie, include_raw=True)
model_structured_output

{
  raw: _ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15', 'langchain-google-genai': '4.3.4'}}, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.1-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x00000275BD8927B0

### Nested Structure

In [3]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    home : str

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    cast: list[Actor]
    genres: list[str] = Field(description="The director of the movie")
    budget: float | None = Field(None, description=" Budget in millions usd")
    
model_structured_output = model.with_structured_output(Movie)
response = model_structured_output.invoke("Please provide the details of the movie 'Inception'")

response

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Movie(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', home='Los Angeles'), Actor(name='Joseph Gordon-Levitt', home='Los Angeles'), Actor(name='Elliot Page', home='Halifax'), Actor(name='Tom Hardy', home='London')], genres=['Science Fiction', 'Action', 'Adventure'], budget=160.0)

### TypedDict

TypeDict provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation .

In [ ]:
'''ChatNVIDIA ddoesn't support'''
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "This year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie rating out of 10"]

TypedDict_output = model.with_structured_output(Movie)
response = TypedDict_output.invoke("Please provide the details of the movie 'Avenger'")

response

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [4]:
from typing_extensions import TypedDict

class Actor(TypedDict):
    name : str
    home : str

class Movie(TypedDict):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    cast: list[Actor]
    genres: list[str] = Field(description="The director of the movie")
    budget: float | None = Field(None, description=" Budget in millions usd")
    
model_structured_output = model.with_structured_output(Movie)
response = model_structured_output.invoke("Please provide the details of the movie 'Avengers'")

response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'home': 'United States'},
  {'name': 'Chris Evans', 'home': 'United States'},
  {'name': 'Mark Ruffalo', 'home': 'United States'},
  {'name': 'Chris Hemsworth', 'home': 'Australia'},
  {'name': 'Scarlett Johansson', 'home': 'United States'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'budget': 220000000}

In [8]:
model.profile

{'name': 'Gemini 3.1 Flash Lite',
 'release_date': '2026-05-07',
 'last_updated': '2026-05-07',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'],
 'reasoning_effort_default': 'minimal'}

### DataClasses
A data class is a class type will clearly containing mainly data although, there are not really any restrictions. you created using @dataclass decorator

In [ ]:
# import os
# os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")

In [1]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gemini-3.1-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])


GoogleAuthError: Unable to find your project. Please provide a project ID by:
- Passing a constructor argument
- Using vertexai.init()
- Setting project using 'gcloud config set project my-project'
- Setting a GCP environment variable
- To create a Google Cloud project, please follow guidance at https://developers.google.com/workspace/guides/create-project

In [2]:
import sys
print(sys.executable)

d:\Users\orayen\LangchainUpdated\.venv\Scripts\python.exe


In [3]:
%pip show langchain-google-vertexai

Note: you may need to restart the kernel to use updated packages.


d:\Users\orayen\LangchainUpdated\.venv\Scripts\python.exe: No module named pip


In [4]:
import sys
!{sys.executable} -m ensurepip --upgrade

Looking in links: c:\Users\nayan\AppData\Local\Temp\tmpwywpq5_6
Processing c:\users\nayan\appdata\local\temp\tmpwywpq5_6\pip-24.3.1-py3-none-any.whl


In [5]:
%pip --version

pip 24.3.1 from d:\Users\orayen\LangchainUpdated\.venv\Lib\site-packages\pip (python 3.13)

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install langchain-google-vertexai

  Using cached langchain_google_vertexai-3.2.4-py3-none-any.whl.metadata (3.3 kB)
  Using cached bottleneck-1.6.0-cp313-cp313-win_amd64.whl.metadata (8.4 kB)
  Using cached google_cloud_aiplatform-1.165.1-py2.py3-none-any.whl.metadata (52 kB)
  Using cached google_cloud_storage-3.17.0-py3-none-any.whl.metadata (14 kB)
  Using cached google_cloud_vectorsearch-0.11.5-py3-none-any.whl.metadata (10.0 kB)
  Using cached numexpr-2.14.2-cp313-cp313-win_amd64.whl.metadata (9.2 kB)
  Using cached pyarrow-23.0.1-cp313-cp313-win_amd64.whl.metadata (3.1 kB)
  Using cached validators-0.36.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached google_api_core-2.42.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached proto_plus-1.29.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached protobuf-6.33.6-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached google_cloud_bigquery-3.46.1-py3-none-any.whl.metadata (8.5 kB)
  Using cached google_cloud_resource_manager-1.19.0-py3-none-any.whl.metadata (9.7 


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
!{sys.executable} -m ensurepip --upgrade

Looking in links: c:\Users\nayan\AppData\Local\Temp\tmpd8ex6e04
